# Tables 1 & 2:  Latent unique-valid table and Accessible/Latent ratio

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.plots.style import apply_paper_style, model_label, model_color, ordered_models
apply_paper_style()


In [ ]:
import os
import pandas as pd

from src.data_loader import load_accessible_professions, load_latent_professions, try_load, warn_incomplete_coverage
from src.metrics.ratios import unique_valid_at_budget, accessible_latent_ratio

accessible = try_load(load_accessible_professions, label="accessible")
latent = try_load(load_latent_professions, expanded=True, label="latent (expanded)")
if accessible is not None:
    warn_incomplete_coverage(accessible, label="accessible")
if latent is not None:
    warn_incomplete_coverage(latent, label="latent")


## Table 1: [Latent] unique valid individuals, model x profession x format, round 100

In [ ]:
table1_wide = None
if latent is None:
    print("Skipping Table 1: no latent data loaded.")
else:
    table1 = unique_valid_at_budget(latent, budget=100, group_cols=["model_version", "profession", "prompt_key"])
    table1_wide = table1.pivot_table(index=["profession", "prompt_key"], columns="model_version", values="unique_valid")
table1_wide


In [ ]:
if table1_wide is None:
    print("Skipping export: Table 1 not computed.")
else:
    out_path = "../data/results/metrics/table1_latent_unique_valid.csv"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    table1_wide.to_csv(out_path)
    print("wrote", out_path)


## Table 2: Accessible-latent ratio, model x profession x format

In [ ]:
table2_wide = None
ratio_df = None
if accessible is None or latent is None:
    missing = [n for n, v in [("accessible", accessible), ("latent", latent)] if v is None]
    print(f"Skipping Table 2: missing {missing}.")
else:
    ratio_df = accessible_latent_ratio(accessible, latent, budget_acc=100, budget_lat=100)
    table2_wide = ratio_df.pivot_table(index=["profession", "prompt_key"], columns="model_version", values="ratio")
    table2_wide = table2_wide.round(2)
table2_wide


In [ ]:
if table2_wide is None:
    print("Skipping export: Table 2 not computed.")
else:
    out_path = "../data/results/metrics/table2_accessible_latent_ratio.csv"
    table2_wide.to_csv(out_path)
    min_ratio = ratio_df["ratio"].min()
    print("wrote", out_path, "| min ratio:", min_ratio)
